In [13]:
!sudo apt-get update -y > /dev/null 2>&1
!sudo apt-get install -y zstd lshw > /dev/null 2>&1
!curl -fsSLs https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import subprocess
import time

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

!ollama pull llama3.2:3b > /dev/null 2>&1
print("Ollama is ready.")

Ollama is ready.


```
langData
| spanish
| | 1000Words.csv
sentence_generations_spanish_1001_..._.csv
sentence_generations_spanish_1001_..._.csv
sentence_generations_spanish_91_..._.csv
sentence_generations_spanish_91_..._.csv
combined_sentence_generations_spanish_1001_..._.zip
combined_sentence_generations_spanish_91_..._.zip
```

In [43]:
import pandas as pd

LANGUAGE = 'spanish'
RUNS = [
    (1,  2),
    (2,  0),
    (3,  0),
    (4,  0),
    (5,  0),
    (6,  0),
    (7,  0),
    (8,  0),
    (9,  0),
    (10, 0),
    (11, 2),
]

WORDS = list(pd.read_csv(f'langData/{LANGUAGE}/1000Words.csv')['foreign_value'].tolist())[:UP_TO_INDEX]

In [45]:
import random
import requests
import re
import csv
import os
import uuid

def generate_sentence(target_words, language):
    # Prepare the prompt
    prompt = f"""
    You are a linguistic assistant.
    REQUIRED WORDS: {", ".join(target_words)}

    TASK: Generate one short sentence/fragment.
    RULES:
    1. You MUST use all of the REQUIRED WORDS.
    2. The sentence must be in the language {language}.

    Output ONLY the sentence.
    """

    response = requests.post('http://localhost:11434/api/generate',
      json={
          "model": "llama3.2:3b",
          "prompt": prompt,
          "stream": False
      })
    return response.json()['response'].strip()

def filter_string(input_str, allowed_set):
    allowed_set = {w.lower().strip() for w in allowed_set}
    words = re.findall(r'\b\w+\b', input_str.lower())
    filtered_words = [word for word in words if word in allowed_set]
    return " ".join(filtered_words)

def validate_and_loop(word_list, language, up_to_index, iterations=5):
    if iterations == 0:
      return

    allowedSet = {w.lower().strip() for w in word_list}
    allowedList = list(allowedSet)
    filename = f'sentence_generations_{language}_{up_to_index}_{uuid.uuid4()}.csv'

    with open(filename, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["generated_sentence"])

    final_sentences = []

    for i in range(iterations):
        target_words = random.sample(allowedList, 3)
        print(f"--- Iteration {i+1} ---")

        current_sentence = generate_sentence(target_words, language)
        filtered_sentence = filter_string(current_sentence, allowedSet)

        with open(filename, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([filtered_sentence])

        final_sentences.append(filtered_sentence)

    return final_sentences

for item in RUNS:
  validate_and_loop(WORDS, LANGUAGE, 7 * 13 * item[0], iterations=item[1])

--- Iteration 1 ---
--- Iteration 2 ---
--- Iteration 1 ---
--- Iteration 2 ---


In [46]:
def translate_text(text):
    prompt = f"Translate the following phrase into English. Output ONLY the translation, nothing else:\n\n{text}"

    try:
        response = requests.post('http://localhost:11434/api/generate',
            json={
                "model": "llama3.2:3b",
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0} # Keep it literal
            })
        return response.json().get('response', '').strip()
    except Exception as e:
        return f"Error: {e}"

def process_translations():
    files = [f for f in glob.glob("*.csv")]

    for f in files:
        df = pd.read_csv(f)

        if 'generated_sentence' not in df.columns:
            print(f"Skipping {f}: Column 'generated_sentence' not found.")
            continue

        translations = []
        for i, row in df.iterrows():
            print(f"  Row {i+1}/{len(df)}", end="\r")
            translations.append(translate_text(row['generated_sentence']))

        df['translated_sentence'] = translations

        output_name = f
        df.to_csv(output_name, index=False)

process_translations()

In [47]:
import glob
import pandas as pd
import shutil
from google.colab import files

def combine_and_download():
    # 1. Identify all CSV files
    csv_files = glob.glob("*.csv")
    groups = {}

    # 2. Categorize files by language and number
    for f in csv_files:
        file_match = re.search(r'sentence_generations_([^_]+)_(\d+)_', f)

        print(f)
        if file_match:
            language = file_match.group(1)
            number = file_match.group(2)
            group_key = f"{language}_{number}"

            if group_key not in groups:
                groups[group_key] = []
            groups[group_key].append(f)

    if not groups:
        print("No matching CSV files found.")
        return

    # 3. Create a clean workspace for results
    output_dir = "combined_results"
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir) # Clear previous runs
    os.makedirs(output_dir, exist_ok=True)

    # 4. Merge and Save
    for group_id, file_list in groups.items():

        # Read all files and stack them
        # ignore_index=True ensures the final CSV has a clean 0 to N row count
        combined_df = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)

        output_filename = os.path.join(output_dir, f"sentence_generations_{group_id}_{uuid.uuid4()}.csv")
        combined_df.to_csv(output_filename, index=False)

    # 5. Archive and Export
    zip_name = "combined_sentence_generation_data"
    shutil.make_archive(zip_name, 'zip', output_dir)

    print(f"Downloading {zip_name}.zip...")
    files.download(f"{zip_name}.zip")

    # 6. Cleanup: Delete all original CSV files and the temporary folder
    for f in csv_files:
        try:
            os.remove(f)
        except OSError as e:
            print(f"Error deleting {f}: {e}")

    # Also delete the 'combined_results' directory and the zip after download
    shutil.rmtree(output_dir)

combine_and_download()

sentence_generations_spanish_1001_80447fd2-1293-44f9-a37b-bb200070c3c9.csv
sentence_generations_spanish_91_7c2a589b-d800-4292-8dc4-054100abd368.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>